# Colab ViT-S/16 DINO: vit_s16_imagenet_covidqu

This notebook runs only `vit_s16_imagenet_covidqu`. It keeps pretraining and fine-tuning in separate cells so you can resume pretraining in epoch chunks.


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone or Pull Repository

In [2]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

!git rev-parse --short HEAD

/content
Cloning into 'contrastive-synthesis-medcls_CVProject'...
remote: Enumerating objects: 21462, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 21462 (delta 125), reused 148 (delta 88), pack-reused 21273 (from 3)
Receiving objects: 100% (21462/21462), 619.95 MiB | 19.92 MiB/s, done.
Resolving deltas: 100% (154/154), done.
Updating files: 100% (21268/21268), done.
/content/contrastive-synthesis-medcls_CVProject
25316011


## 3. Install Minimal Dependencies

In [3]:
import importlib.util
import subprocess
import sys

packages = {
    'timm': 'timm',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'pandas': 'pandas',
    'PIL': 'Pillow',
}

to_install = []
for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        to_install.append(package_name)

if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
else:
    print('All required packages already installed.')

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi

All required packages already installed.
Torch: 2.11.0+cu128
CUDA available: True
Mon Jun  1 14:24:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |               

## 4. Editable Paths and Run Flags

`PRETRAIN_EPOCH_OVERRIDE` is the total target DINO epoch count. If a `last_dino_checkpoint.pth` already exists in the output folder, pretraining resumes up to this total.

`LOCAL_CROPS_NUMBER=4` uses DINO multi-crop. If a `timm` version still rejects 96x96 crops, set it to `0` and rerun pretraining.


In [11]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments')
SYNTHETIC_MANIFEST = Path('/content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv')
REAL_UNLABELED_DIR = REPO_ROOT / 'data/processed/unlabelled_16934'

RUN_VIT_COVIDQU = False
RUN_VIT_IMAGENET_COVIDQU = True
RUN_VIT_COVIDQU_SYN = False
RUN_VIT_IMAGENET_COVIDQU_SYN = False

# Run pretraining in small chunks by increasing this total target: 10, 20, 30, ...
RUN_PRETRAIN = True
RUN_FINETUNE = True  # Set True only after the chosen pretraining target is finished.
PRETRAIN_EPOCH_OVERRIDE = 100
FINETUNE_EPOCH_OVERRIDE = None
LOCAL_CROPS_NUMBER = 4  # DINO default here. If timm crop-size errors persist, set this to 0.

pretrain_epoch_arg = '' if PRETRAIN_EPOCH_OVERRIDE is None else f'--epochs {PRETRAIN_EPOCH_OVERRIDE}'
local_crops_arg = f'--local-crops-number {LOCAL_CROPS_NUMBER}'
finetune_epoch_arg = '' if FINETUNE_EPOCH_OVERRIDE is None else f'--epochs {FINETUNE_EPOCH_OVERRIDE}'

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST, SYNTHETIC_MANIFEST.exists())
print('REAL_UNLABELED_DIR:', REAL_UNLABELED_DIR, REAL_UNLABELED_DIR.exists())
print('RUN_PRETRAIN:', RUN_PRETRAIN)
print('RUN_FINETUNE:', RUN_FINETUNE)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('local_crops_arg:', local_crops_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)

/content/contrastive-synthesis-medcls_CVProject
OUTPUT_ROOT: /content/drive/MyDrive/medcls_cvproject/results/experiments
SYNTHETIC_MANIFEST: /content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv True
REAL_UNLABELED_DIR: /content/contrastive-synthesis-medcls_CVProject/data/processed/unlabelled_16934 True
RUN_PRETRAIN: True
RUN_FINETUNE: True
pretrain_epoch_arg: --epochs 100
local_crops_arg: --local-crops-number 4
finetune_epoch_arg: 


## 5. Verify Inputs and Scripts

In [7]:
!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_dino_vit.py scripts/run_classification_vit.py
!python scripts/run_dino_vit.py --help | grep resume || true
!python scripts/run_classification_vit.py --help | grep pretrained || true


Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

## 6. Experiment: vit_s16_imagenet_covidqu

DINO pretraining from ImageNet initialization on real unlabeled COVID-QU, then supervised fine-tuning on real labeled manifests.

Run the pretraining cell repeatedly by increasing `PRETRAIN_EPOCH_OVERRIDE` from 10 to 20, 30, and so on. Run the fine-tuning cell only after pretraining reaches the target you want to report.

### 6a. Pretrain Only: vit_s16_imagenet_covidqu

In [8]:
EXP = 'vit_s16_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_dino_checkpoint.pth'

if RUN_VIT_IMAGENET_COVIDQU and RUN_PRETRAIN:
    !python scripts/run_dino_vit.py \
      --config configs/experiments/vit_s16/imagenet_covidqu.yaml \
      --real-unlabeled-dir "{REAL_UNLABELED_DIR}" \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {local_crops_arg} \
      {pretrain_epoch_arg}
else:
    print('Skipping pretrain', EXP)


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Resuming DINO from /content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_imagenet_covidqu/pretrain/checkpoints/last_dino_checkpoint.pth at epoch 10
Epoch 11/100 dino_loss=2.1384
Epoch 12/100 dino_loss=2.1138
Epoch 13/100 dino_loss=2.0445
Epoch 14/100 dino_loss=1.9914
Epoch 15/100 dino_loss=1.9272
Epoch 16/100 dino_loss=1.8451
Epoch 17/100 dino_loss=1.7990
Epoch 18/100 dino_loss=1.7620
Epoch 19/100 dino_loss=1.7249
Epoch 20/100 dino_loss=1.6835
Epoch 21/100 dino_loss=1.6355
Epoch 22/100 dino_loss=1.5874
Epoch 23/100 dino_loss=1.5462
Epoch 24/100 dino_loss=1.5041
Epoch 25/100 dino_loss=1.4684
Epoch 26/100 dino_loss=1.4321
Epoch 27/100 dino_loss=1.3897
Epoch 28/100 dino_loss=1.3551
Epoch 29/100 dino_loss=1.3248
Epoch 30/100 dino_loss=1.2954
Epoch 31

### 6b. Fine-Tune Only: vit_s16_imagenet_covidqu

In [12]:
EXP = 'vit_s16_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'

if RUN_VIT_IMAGENET_COVIDQU and RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'DINO checkpoint not found: {CKPT}. Finish pretraining first.')
    !python scripts/run_classification_vit.py \
      --config configs/experiments/vit_s16/imagenet_covidqu.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {finetune_epoch_arg}
else:
    print('Skipping finetune', EXP)


Missing keys after DINO teacher load: ['head.weight', 'head.bias']
Epoch 1/50 train_loss=0.7609 val_loss=0.5000 val_acc=0.8409 val_f1_macro=0.8237
Epoch 2/50 train_loss=0.3364 val_loss=0.3698 val_acc=0.8812 val_f1_macro=0.8730
Epoch 3/50 train_loss=0.2191 val_loss=0.3217 val_acc=0.8979 val_f1_macro=0.8935
Epoch 4/50 train_loss=0.1529 val_loss=0.2932 val_acc=0.9050 val_f1_macro=0.9058
Epoch 5/50 train_loss=0.1108 val_loss=0.2803 val_acc=0.9074 val_f1_macro=0.9076
Epoch 6/50 train_loss=0.0767 val_loss=0.2706 val_acc=0.9074 val_f1_macro=0.9072
Epoch 7/50 train_loss=0.0552 val_loss=0.2641 val_acc=0.9169 val_f1_macro=0.9164
Epoch 8/50 train_loss=0.0374 val_loss=0.2995 val_acc=0.9097 val_f1_macro=0.9093
Epoch 9/50 train_loss=0.0251 val_loss=0.2738 val_acc=0.9145 val_f1_macro=0.9144
Epoch 10/50 train_loss=0.0170 val_loss=0.2980 val_acc=0.9169 val_f1_macro=0.9163
Epoch 11/50 train_loss=0.0120 val_loss=0.2944 val_acc=0.9169 val_f1_macro=0.9169
Epoch 12/50 train_loss=0.0104 val_loss=0.3073 val_a

## 7. Display Result


In [10]:
import json
import pandas as pd

EXP = 'vit_s16_imagenet_covidqu'
metrics_path = OUTPUT_ROOT / EXP / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('No metrics found yet:', metrics_path)

!find "{OUTPUT_ROOT}/vit_s16_imagenet_covidqu" -maxdepth 4 -type f \( -name 'metrics.json' -o -name 'best_dino_teacher.pth' -o -name 'last_dino_checkpoint.pth' -o -name 'confusion_matrix.png' -o -name 'classification_report.csv' \) | sort


No metrics found yet: /content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_imagenet_covidqu/metrics.json
/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_imagenet_covidqu/pretrain/checkpoints/best_dino_teacher.pth
/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_imagenet_covidqu/pretrain/checkpoints/last_dino_checkpoint.pth
